## CI workflow — promotion dev → test → prod

The end-to-end shape, with GitHub Actions as the runner:

```text
  developer commits ─► push ─► PR opened
                                 │
                                 ▼
              GH Actions: validate + lint
                • bundle validate -t dev / test / prod
                                 │
                 PR review + merge to main
                                 ▼
              GH Actions: deploy
                • bundle deploy -t test
                • run smoke job, wait for green
                • bundle deploy -t prod
```

A minimal `.github/workflows/deploy.yml`:

```yaml
name: Deploy bundle
on: { push: { branches: [main] } }
jobs:
  deploy:
    runs-on: ubuntu-latest
    env:
      DATABRICKS_HOST:          ${{ secrets.DATABRICKS_HOST }}
      DATABRICKS_CLIENT_ID:     ${{ secrets.DATABRICKS_CLIENT_ID }}
      DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
    steps:
      - uses: actions/checkout@v4
      - uses: databricks/setup-cli@main
      - run: databricks bundle deploy -t test
      - run: databricks bundle run smoke_test_job -t test
      - run: databricks bundle deploy -t prod
```

**The exam expects you to recognise** the split: **`validate` runs on every PR**; **`deploy` runs on merge to main**, scoped per target, usually gated behind a smoke test before prod. Both commands are **identical regardless of cloud or workspace** — the target manifest changes; the CLI does not.
